# Roll B: Data Cleaning — Andmete puhastamine ja valideerimine
**Nädal 7 | UrbanStyle RFM analüüs**  
**Autor:** Vladimir G  
**Sisend:** Roll A väljund — liidatud DataFrame (df)

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Laadi Roll A väljund (liidatud DataFrame)
df_sales = pd.read_csv('sales.csv')
df_customers = pd.read_csv('customers.csv')
df = pd.merge(
    df_sales,
    df_customers[['customer_id', 'email', 'first_name', 'last_name', 'city']],
    on='customer_id',
    how='left'
)
print(f'Esialgne shape: {df.shape}')

In [ ]:
# 1. Duplikaadid
dupes = df.duplicated(subset='sale_id').sum()
print(f'Duplikaadid (sale_id alusel): {dupes}')
df = df.drop_duplicates(subset='sale_id')
print(f'Pärast duplikaatide eemaldamist: {df.shape}')

In [ ]:
# 2. NULL väärtused
print('NULL väärtused veergude kaupa:')
print(df.isnull().sum())

df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])
print(f'\nPärast NULL eemaldamist: {df.shape}')

In [ ]:
# 3. Kuupäevade parsimine (mixed formats: YYYY-MM-DD ja DD/MM/YYYY)
def parse_date_mixed(d):
    try:
        if '/' in str(d):
            return pd.to_datetime(d, format='%d/%m/%Y')
        return pd.to_datetime(d, format='%Y-%m-%d')
    except:
        return pd.NaT

df['sale_date'] = df['sale_date'].apply(parse_date_mixed)
df = df.dropna(subset=['sale_date'])
print(f'sale_date dtype: {df["sale_date"].dtype}')
print(f'Kuupäevavahemik: {df["sale_date"].min().date()} kuni {df["sale_date"].max().date()}')

In [ ]:
# 4. Negatiivsed hinnad (tagastused) ja tuleviku kuupäevad
neg_count = (df['total_price'] <= 0).sum()
print(f'Negatiivsed/null total_price: {neg_count}')
df = df[df['total_price'] > 0]

reference_date = pd.Timestamp('2025-02-28')
future = (df['sale_date'] > reference_date).sum()
print(f'Tuleviku kuupäevad: {future}')
df = df[df['sale_date'] <= reference_date]

# Puhastusraport
print('\n=== PUHASTUSRAPORT ===')
print(f'Lõplik shape:        {df.shape}')
print(f'Unikaalseid kliente: {df["customer_id"].nunique()}')
print(f'Kuupäevavahemik:     {df["sale_date"].min().date()} kuni {df["sale_date"].max().date()}')
print(f'Käive kokku:         {df["total_price"].sum():,.2f} EUR')
print(f'Keskmine tellimus:   {df["total_price"].mean():,.2f} EUR')